In [3]:
import re
import pandas as pd
import nltk
from nltk.translate.meteor_score import meteor_score
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

# 1. LOAD DATA FROM URL
url = "https://raw.githubusercontent.com/johnsyin/employee-review-sentiment-dashboard/master/data/employee_reviews.csv"
data = pd.read_csv(url, index_col=0)
data["review"] = data["summary"].fillna("") + " " + data["pros"].fillna("") + " " + data["cons"].fillna("")
data["rating"] = pd.to_numeric(data["overall-ratings"], errors="coerce")
data = data.dropna(subset=["rating"])
data["label"] = data["rating"].apply(lambda r: "negative" if r <= 2 else ("neutral" if r == 3 else "positive"))
data = data.groupby("label").sample(n=3000, random_state=42)

# 2. TEXT PROCESSOR + STOPWORD REMOVAL
keep = {"not", "no", "never", "nor", "neither", "nothing"}
stop_words = set(ENGLISH_STOP_WORDS) - keep

def clean_text(text):
    text = re.sub(r"[^a-z\s]", " ", text.lower())
    return " ".join(w for w in text.split() if w not in stop_words and len(w) > 1)

# 3. TF-IDF + NAIVE BAYES
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=20000)
X = tfidf.fit_transform(data["review"].apply(clean_text))
model = MultinomialNB(alpha=0.5).fit(X, data["label"])

# 4. LEXICON SENTIMENT ANALYSIS (used as the reference for METEOR)
good = {"great","excellent","good","love","amazing","fantastic","wonderful","best","happy",
        "supportive","friendly","helpful","awesome","nice","enjoy","enjoyable","recommend"}
bad = {"terrible","awful","horrible","bad","worst","toxic","rude","poor","low","boring",
       "stress","stressful","hate","unfair","underpaid","lazy","stressed","hostile"}

def lexicon_sentiment(text):
    words = re.sub(r"[^a-z\s]", " ", text.lower()).split()
    score = 0
    for i, w in enumerate(words):
        p = 1 if w in good else -1 if w in bad else 0
        if p and i > 0 and words[i-1] in {"not", "no", "never"}:   # negation flips
            p = -p
        score += p
    return "positive" if score > 0 else "negative" if score < 0 else "neutral"

# 5. USER INPUT -> PREDICT -> METEOR (for every input)
print("Model trained!\n")
while True:
    review = input("Enter a job review (or 'quit'): ").strip()
    if review.lower() == "quit":
        break

    pred = model.predict(tfidf.transform([clean_text(review)]))[0]
    ref_label = lexicon_sentiment(review)

    ref = f"this review is {ref_label}".split()
    hyp = f"this review is {pred}".split()

    print("Predicted sentiment:", pred.upper())
    print(f"Lexicon reference  : {ref_label}")
    print(f"METEOR score       : {meteor_score([ref], hyp):.3f}\n")

Model trained!



Enter a job review (or 'quit'):  The manager is not helpful and not friendly


Predicted sentiment: NEGATIVE
Lexicon reference  : negative
METEOR score       : 0.992



Enter a job review (or 'quit'):  quit
